In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

# --------------------------------------------------------------------
# 0. Configuration ----------------------------------------------------
model_name  = "open-r1/Qwen2.5-Math-7B-RoPE-300k"
split       = "train[:2]"                          # small sample
chat_tpl    = """{%- if tools %}{# ... entire Jinja template you pasted ... #}{%- endif %}"""
# --------------------------------------------------------------------

# 1. Dataset ----------------------------------------------------------
ds = load_dataset("open-r1/Mixture-of-Thoughts", "all", split=split)

# 2. Apply chat template (pure python, no trl) ------------------------
from jinja2 import Template

def render_chat(example):
    # You have the template already as a string; Jinja2 compiles it.
    rendered = Template(chat_tpl).render(
        messages=example["messages"],
        tools=None, 
        add_generation_prompt=False)
    return {"text": rendered}

ds = ds.map(render_chat, desc="rendering chat template")

# 3. Tokenise ---------------------------------------------------------
tok = AutoTokenizer.from_pretrained(model_name)
ds  = ds.map(lambda ex: tok(text=ex["text"], add_special_tokens=True),
             desc="tokenising")

# 4. Collate + build labels ------------------------------------------
collator = DataCollatorForLanguageModeling(
    pad_token_id=tok.pad_token_id,
    mlm=False,                         # make sure it behaves as LM collator
)

batch = collator([ds[0]])              # single example for inspection
ids, labels = batch["input_ids"][0], batch["labels"][0]

# 5. Inspect ----------------------------------------------------------
tokens_readable = tok.convert_ids_to_tokens(ids.tolist())
labels_readable = [
    tok.convert_ids_to_tokens([t])[0] if t != -100 else "PAD"
    for t in labels.tolist()
]

print("First 40 input tokens :", tokens_readable[:40])
print("First 40 label tokens :", labels_readable[:40])


KeyboardInterrupt: 